In [ ]:
REPO <- "https://raw.githubusercontent.com/vicleme/PI3-FFV/main/"   # o endereco Raw do repositorio do grupo (secao 1 da ficha)
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")   # funcoes do curso, ate a Aula 07
cfg <- tryCatch({                                                            # tenta o config.R do grupo...
  download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R", quiet = TRUE)
  source("config.R"); cfg                                                    # ...e usa as decisoes do grupo
}, error = function(e) {                                                     # sem config.R no repositorio (404):
  message("Sem config.R no repositorio do grupo: usando as decisoes padrao do curso.")
  cfg_aula()                                                                 # as decisoes canonicas do motor
})
if (!file.exists("config.R"))                                                # se nao veio do repositorio,
  writeLines(c("cfg <- ", capture.output(dput(cfg))), "config.R")           #   grava um config.R para o grupo
cp     <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"),      # o corpus CONGELADO na Aula 05
                   stringsAsFactors = FALSE, fileEncoding = "UTF-8")         #   texto fica texto; acentos em UTF-8
docs   <- setNames(cp$texto, cp$id)                                          # vetor nomeado id -> texto
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))   # de que artigo veio cada documento
ix     <- montar(docs, cfg)                                                  # tudo derivado: indice, TF-IDF, BM25
estado()                                                                     # a tutora compara com a secao 8 da ficha

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-11-26 18:57 · R version 4.6.1 (2026-06-24) · motor08 v2 2026-10-22 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `a` | character | 1 | qrels_consenso.csv |
| `ap0` | numeric, nomeado | 7 | q02 0.5, q03 1, q05 0.07692, q06 1, q07 0.3667, q09 0.5, ... |
| `ap1` | numeric, nomeado | 7 | q02 1, q03 1, q05 0.07692, q06 1, q07 0.5, q09 0.5, ... |
| `arqs` | character | 1 | qrels_consenso.csv |
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 203 | a, ao, aos, aquela, aquelas, aquele, ... |
| `cfg$k1` | numeric | 1 | 1.2 |
| `cfg$b` | numeric | 1 | 0.75 |
| `cfg$limiar` | numeric | 1 | 2 |
| `cfg$juizes` | character | 1 | consenso |
| `cfg$prf_k` | numeric | 1 | 0 |
| `cfg$rocchio` | numeric, nomeado | 3 | a 1, b 0.75, g 0.15 |
| `cfg$modelo` | c

In [ ]:
arqs <- c("qrels_consenso.csv")                                             # o gabarito do grupo (juiz = consenso)
for (a in arqs) download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a)   # traz o qrels para o Colab
qrels <- ler_qrels(arqs, juiz = cfg$juizes)                                 # le, tira BOM e _p2; prioridade da Aula 5,5
nec   <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"),     # as necessidades da Aula 05,
                  stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")     #   com a coluna conjunto (dev / teste)
consultas <- setNames(nec$texto_consulta, nec$consulta)                     # q01 -> o texto digitado
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))                     # relevantes por consulta, com o limiar do grupo
usar <- names(qrels)[R_q > 0]                                               # as que entram na avaliacao
names(qrels)[R_q == 0]                                                      # AVISO: estas ficam fora (R = 0), como na Aula 5,5

character(0)

In [ ]:
names(qrels); length(usar); R_q

[1] "q01" "q02" "q03" "q04" "q05" "q06" "q07" "q08" "q09" "q10"

[1] 10

q01 q02 q03 q04 q05 q06 q07 q08 q09 q10 
  1   1   3   1   1   1   2   1   2   1

In [ ]:
proximidade <- function(qt, dt, janela = 3) {        # qt: tokens da consulta; dt: tokens do documento
  s <- 0                                             # o escore comeca em zero
  for (i in seq_along(qt)) for (j in seq_along(qt))  # todos os pares de posicoes da consulta...
    if (i < j && qt[i] != qt[j]) {                   # ...com i antes de j e termos diferentes
      pos_i <- which(dt == qt[i])                    # onde o termo i aparece no documento
      pos_j <- which(dt == qt[j])                    # onde o termo j aparece
      if (length(pos_i) && length(pos_j)) {          # os dois aparecem?
        d <- outer(pos_j, pos_i, "-")                # todas as distancias (j menos i)
        d <- d[d > 0 & d <= janela]                  # so j DEPOIS de i, e dentro da janela
        if (length(d)) s <- s + 1 / min(d)           # soma 1 / a menor distancia
      }
    }
  s                                                  # devolve o escore
}
reordenar <- function(rk, k, sc) {               # rk: ranking do estagio 1; sc: escores do estagio 2
  k <- min(k, length(rk))                        # protecao: k nao passa do numero de documentos
  top <- names(rk)[1:k]                          # a cabeca: os k primeiros
  nova <- top[order(-sc[top], seq_along(top))]   # pelo escore, decrescente; empate: ordem do estagio 1
  c(nova, names(rk)[-(1:k)])                     # cabeca reordenada + cauda intacta (NOMES)
}
proximidade(c("a", "b"), c("a", "x", "b"))                         # teste: "b" 2 depois de "a"
reordenar(c(a = 3, b = 2, c = 1), 2, c(a = 0.1, b = 0.9, c = 5))   # teste da reordenacao

[1] 0.5

[1] "b" "a" "c"

In [ ]:
proximidade <- function(qt, dt, janela = 3) {        # qt: tokens da consulta; dt: tokens do documento
  s <- 0                                             # o escore comeca em zero
  for (i in seq_along(qt)) for (j in seq_along(qt))  # todos os pares de posicoes da consulta...
    if (i < j && qt[i] != qt[j]) {                   # ...com i antes de j e termos diferentes
      pos_i <- which(dt == qt[i])                    # onde o termo i aparece no documento
      pos_j <- which(dt == qt[j])                    # onde o termo j aparece
      if (length(pos_i) && length(pos_j)) {          # os dois aparecem?
        d <- outer(pos_j, pos_i, "-")                # todas as distancias (j menos i)
        d <- d[d > 0 & d <= janela]                  # so j DEPOIS de i, e dentro da janela
        if (length(d)) s <- s + 1 / min(d)           # soma 1 / a menor distancia
      }
    }
  s                                                  # devolve o escore
}
reordenar <- function(rk, k, sc) {               # rk: ranking do estagio 1; sc: escores do estagio 2
  k <- min(k, length(rk))                        # protecao: k nao passa do numero de documentos
  top <- names(rk)[1:k]                          # a cabeca: os k primeiros
  nova <- top[order(-sc[top], seq_along(top))]   # pelo escore, decrescente; empate: ordem do estagio 1
  c(nova, names(rk)[-(1:k)])                     # cabeca reordenada + cauda intacta (NOMES)
}
proximidade(c("a", "b"), c("a", "x", "b"))                         # teste: "b" 2 depois de "a"
reordenar(c(a = 3, b = 2, c = 1), 2, c(a = 0.1, b = 0.9, c = 5))   # teste da reordenacao

[1] 0.5

[1] "b" "a" "c"

In [ ]:
# sem a decisao da Aula 07 ou da 08 no config.R: os valores padrao do curso
if (is.null(cfg$modelo)) cfg$modelo <- "bm25"   # estagio 1 = BM25 (quem nao fez a Aula 07)
if (is.null(cfg$lsa_k))  cfg$lsa_k  <- 2        # so usado com modelo "lsa" ou "hibrido"
if (is.null(cfg$rrf_c))  cfg$rrf_c  <- 60       # so usado com modelo "hibrido"
if (is.null(cfg$janela)) cfg$janela <- 3        # janela da proximidade (Aula 08)
estagio1 <- function(texto, ix, cfg, m) {                  # o ranking que o motor do grupo da hoje
  rb <- ranking_bm25(texto, ix, cfg)                       # o BM25, com k1 e b do grupo
  if (cfg$modelo == "bm25") return(rb)                     # modelo "bm25": pronto
  rl <- ranking_lsa(texto, ix, cfg, m)                     # o LSA da Aula 07 (m = lsa(ix, k))
  if (cfg$modelo == "lsa") rl else rrf(rb, rl, c = cfg$rrf_c)   # "lsa", ou o hibrido pela RRF
}                                                          # fim da funcao
m <- if (cfg$modelo == "bm25") NULL else lsa(ix, cfg$lsa_k)   # o LSA so e calculado se for usado
rerank_prox <- function(rk, qt, tokens, k, janela) {    # rk: estagio 1; qt: tokens da consulta
  if (k == 0) return(names(rk))                         # k = 0: nao reordena
  top <- head(names(rk), k)                             # os candidatos (head corta se k > N)
  sc  <- sapply(tokens[top], function(dt) proximidade(qt, dt, janela))   # so eles sao lidos
  reordenar(rk, k, sc)                                  # cabeca reordenada + cauda
}                                                       # fim da funcao

In [ ]:
q  <- "q03"                                                 # consulta de desenvolvimento (nao use q01, q04, q08)
rk <- estagio1(consultas[[q]], ix, cfg, m)                  # estagio 1: o BM25 do grupo
qt <- preparar(consultas[[q]], cfg); qt                     # os tokens da consulta, COM as stopwords do grupo
round(sapply(ix$tokens[names(rk)[1:10]], function(dt) proximidade(qt, dt, 3)), 3)   # prox do top-10, na ordem do estagio 1
rn <- rerank_prox(rk, qt, ix$tokens, 10, 3)                 # reordena os 10 primeiros, janela 3
rbind(estagio1 = names(rk)[1:5], rerank = rn[1:5])          # os dois top-5 lado a lado
docs[rn[1:5][!rn[1:5] %in% names(rk)[1:5]]]                 # os que SUBIRAM para o top-5: leiam

[1] "maior"   "porto"   "america" "latina"

Santos_p19  Santos_p18   Santos_p1  Santos_p17  Santos_p22 Guaruja_p18 
      1.000       4.333       4.333       0.000       0.000       0.000 
  Santos_p9  Santos_p10 Cubatao_p18   Santos_p6 
      0.000       0.000       0.000       0.000

estagio1,Santos_p19,Santos_p18,Santos_p1,Santos_p17,Santos_p22
rerank,Santos_p18,Santos_p1,Santos_p19,Santos_p17,Santos_p22


named character(0)

In [ ]:
docs[c("Santos_p19", "Santos_p18", "Santos_p1")]   # o que desceu (p19) e os dois que subiram

Santos_p19 
                                                                                                                                                    "A área de influência econômica do porto concentra mais de 50% do produto interno bruto do país e abrange principalmente os estados de São Paulo, Minas Gerais, Goiás, Mato Grosso e Mato Grosso do Sul. Aproximadamente 90% da base industrial paulista está localizada a menos de 200 quilômetros do porto santista. Em 2007, o Porto de Santos foi considerado o 39ª maior do mundo por movimentação de contêineres, sendo o mais movimentado da América Latina." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              Santos_p18 
                    "Santos possui o maior porto da América Latina, o Porto de Santos. Suas obras começaram em 1888 e o primeiro trecho de cais foi inaugurado em 1892. O porto de Santos é responsável por escoar boa parte das exportações brasileiras e cerca de 70% das exportações de café. O Complexo Portuário de Santos responde por mais de um quarto da movimentação da balança comercial brasileira e é o principal porto do país, incluindo na pauta de suas principais cargas produtos como o açúcar, soja, café, milho, trigo, sal, suco de laranja, papel, automóveis, álcool e outros granéis líquidos." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               Santos_p1 
"Santos é um município brasileiro no litoral do estado de São Paulo, Região Sudeste do país. Abriga o maior porto da América Latina, que é o principal responsável pela dinâmica econômica da cidade ao lado do turismo, da pesca e do comércio. Ocupa a 5.ª colocação entre as não capitais mais importantes para a economia brasileira e 10.ª colocada segundo a qualidade de vida. A cidade é sede do poder executivo paulista todo dia 13 de junho e não apenas sede de diversas instituições de ensino superior como também da mais antiga entidade geral estudantil do Brasil, o Centro dos Estudantes de Santos."

In [ ]:
rel <- names(qrels$q03)[qrels$q03 >= cfg$limiar]            # os relevantes da q03 (grau >= limiar do grupo)
d   <- union(rel, c("Santos_p19", "Santos_p18", "Santos_p1"))   # os relevantes + os tres que lemos
data.frame(doc = d, grau = qrels$q03[d],                    # o grau no gabarito do grupo
           estagio1 = match(d, names(rk)),                  # posicao no BM25
           rerank   = match(d, rn),                         # posicao depois da reordenacao (k = 10)
           row.names = NULL)                                # tabela limpa, sem nomes nas linhas

doc,grau,estagio1,rerank
<chr>,<dbl>,<int>,<int>
Santos_p1,2,3,2
Santos_p18,2,2,1
Santos_p19,2,1,3


In [ ]:
ap_rerank <- function(rk, qt, grau, tokens, k, janela, limiar)               # AP de UMA consulta
  metricas(rerank_prox(rk, qt, tokens, k, janela), grau, limiar)[["AP"]]     # o AP da ordem final
dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])                # aqui se escolhe
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])              # so no fim, uma vez
rks <- lapply(consultas[usar], function(texto) estagio1(texto, ix, cfg, m))  # o estagio 1 de cada consulta, uma vez
qts <- lapply(consultas[usar], function(texto) preparar(texto, cfg))         # os tokens de cada consulta
sapply(qts, length)                                                          # quantos termos sobraram em cada uma

q01 q02 q03 q04 q05 q06 q07 q08 q09 q10 
  3   3   4   2   2   4   4   3   3   4

In [ ]:
map0 <- mean(sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 0, 3, cfg$limiar)))
round(map0, 3)                                                    # o MAP do estagio 1 sozinho (k = 0)
grade <- sapply(c(2, 3, 5), function(w)                           # cada janela vira uma coluna...
  sapply(c(3, 5, 10), function(k)                                 # ...cada k, uma linha
    mean(sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, k, w, cfg$limiar)))))
rownames(grade) <- c("k3", "k5", "k10"); colnames(grade) <- c("janela2", "janela3", "janela5")
round(grade, 3)                                                   # o MAP de cada combinacao

[1] 0.635

,janela2,janela3,janela5
k3,0.635,0.635,0.706
k5,0.654,0.654,0.725
k10,0.654,0.654,0.718


In [ ]:
ap0 <- sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 0, 5, cfg$limiar))   # estagio 1 (k = 0)
ap1 <- sapply(dev, function(q) ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, 5, 5, cfg$limiar))   # k = 5, janela 5 (troquem)
round(rbind(estagio1 = ap0, rerank = ap1), 3)                     # consulta a consulta
dev[ap1 < ap0]                                                    # onde a reordenacao perdeu
dev[ap1 > ap0]                                                    # onde a reordenacao ganhou

,q02,q03,q05,q06,q07,q09,q10
estagio1,0.5,1,0.077,1,0.367,0.5,1
rerank,1.0,1,0.077,1,0.500,0.5,1


character(0)

[1] "q02" "q07"

In [ ]:
q <- "q02"                                                          # a consulta que mais subiu
qts[[q]]                                                            # os termos da consulta
rbind(estagio1 = names(rks[[q]])[1:5],                              # top-5 do BM25...
      rerank   = rerank_prox(rks[[q]], qts[[q]], ix$tokens, 5, 5)[1:5])   # ...e depois da reordenacao
docs[c("Santos_p14", "Guaruja_p2")]                                 # o relevante e o que estava na frente: leiam
data.frame(termo = qts[[q]],                                        # onde cada termo cai no Santos_p14
           posicoes = sapply(qts[[q]], function(t) paste(which(ix$tokens$Santos_p14 == t), collapse = ",")),
           row.names = NULL)                                        # tabela limpa

[1] "praias"     "santos"     "quantidade"

estagio1,Santos_p11,Santos_p14,Guaruja_p2,Santos_p12,Guaruja_p17
rerank,Santos_p14,Santos_p11,Guaruja_p2,Santos_p12,Guaruja_p17


Santos_p14 
                                                                                                                                                                                                                                                                                                               "A orla santista é composta por 6 praias em oito quilômetros de extensão. Santos possui clima tropical litorâneo úmido. Os verões são quentes e úmidos, enquanto os invernos têm como característica temperaturas mais amenas e menor incidência de chuvas." 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 Guaruja_p2 
"Atualmente, a cidade de Guarujá é conhecida como a Pérola do Atlântico, devido às suas belas praias e belezas naturais. Muito procurada pelos turistas na alta temporada, a cidade conta com praias urbanizadas e algumas selvagens, acessíveis apenas por trilhas. Além do litoral, Guarujá oferece construções históricas e trilhas de ecoturismo. Outra atração local é a pesca artesanal, que pode ser vista e praticada em diversas praias do município ao longo de sua orla. Anualmente, a cidade recebe mais de 1,5 milhão de turistas de diversas partes do país."

termo,posicoes
<chr>,<chr>
praias,5
santos,9
quantidade,


In [ ]:
r5 <- names(qrels$q05)[qrels$q05 >= cfg$limiar]   # o(s) relevante(s) da q05
match(r5, names(rks$q05))                         # posicao dele(s) no estagio 1 (BM25)

[1] 13

In [ ]:
k_esc <- 5; j_esc <- 5                                            # O QUE O GRUPO ESCOLHEU
tab_teste <- sapply(teste, function(q)                            # para cada consulta de teste...
  sapply(c(estagio1 = 0, rerank = k_esc), function(k)             # ...a referencia (k = 0) e a escolha
    ap_rerank(rks[[q]], qts[[q]], qrels[[q]], ix$tokens, k, j_esc, cfg$limiar)))
round(tab_teste, 3); round(rowMeans(tab_teste), 3)                # relatar como saiu

,q01,q04,q08
estagio1,0.056,0.333,0.062
rerank,0.056,0.333,0.062


estagio1   rerank 
    0.15     0.15

In [ ]:
source("config.R")                                                         # recarrega com os campos novos
cfg$rerank_k; cfg$janela                                                   # confiram com a decisao
ranking_final <- function(texto, ix, cfg, m)                               # o motor inteiro, de um texto
  rerank_prox(estagio1(texto, ix, cfg, m), preparar(texto, cfg),           # estagio 1 do cfg$modelo...
              ix$tokens, cfg$rerank_k, cfg$janela)                         # ...e o estagio 2 do cfg
head(ranking_final(consultas[["q03"]], ix, cfg, m), 5)                     # o top-5 que o motor daria na q03

[1] 5

[1] 5

[1] "Santos_p18" "Santos_p1"  "Santos_p19" "Santos_p17" "Santos_p22"

In [ ]:
grade_q07 <- sapply(c(2, 3, 5), function(w)
  sapply(c(3, 5, 10), function(k)
    ap_rerank(rks[["q07"]], qts[["q07"]], qrels[["q07"]], ix$tokens, k, w, cfg$limiar)))
rownames(grade_q07) <- c("k3", "k5", "k10")
colnames(grade_q07) <- c("janela2", "janela3", "janela5")
round(grade_q07, 3)

,janela2,janela3,janela5
k3,0.367,0.367,0.367
k5,0.500,0.500,0.500
k10,0.500,0.500,0.450


In [ ]:
q7 <- "q07"
qt7 <- qts[[q7]]
rk7 <- rks[[q7]]
top10_7 <- names(rk7)[1:10]
prox7 <- round(sapply(ix$tokens[top10_7], function(dt) proximidade(qt7, dt, 5)), 3)

rn7_k5  <- rerank_prox(rk7, qt7, ix$tokens, 5, 5)
rn7_k10 <- rerank_prox(rk7, qt7, ix$tokens, 10, 5)

data.frame(
  pos_bm25 = 1:10,
  doc      = top10_7,
  grau     = qrels[[q7]][top10_7],
  prox_w5  = prox7,
  pos_k5   = match(top10_7, rn7_k5),
  pos_k10  = match(top10_7, rn7_k10),
  row.names = NULL
)

pos_bm25,doc,grau,prox_w5,pos_k5,pos_k10
<int>,<chr>,<dbl>,<dbl>,<int>,<int>
1,Santos_p15,0,0.00,3,4
2,Cubatao_p1,0,1.00,1,1
3,Cubatao_p8,2,0.00,4,5
4,Cubatao_p4,0,0.00,5,6
5,Cubatao_p9,2,0.50,2,2
6,Cubatao_p18,0,0.25,6,3
7,Cubatao_p20,0,0.00,7,7
8,Guaruja_p20,0,0.00,8,8
9,Cubatao_p3,0,0.00,9,9


In [ ]:
anexar_estado("00_FICHA_PROJETO.md")

Estado do R anexado a 00_FICHA_PROJETO.md (95 linhas de objetos).



ERROR: Error: arquivo não encontrado: aula08_parteD_consolidado.md
Salve primeiro o consolidado que a tutora gerou, depois rode de novo.
Você está em: /content


In [ ]:
anexar_estado("aula08_parteD_consolidado.md")

Estado do R anexado a aula08_parteD_consolidado.md (95 linhas de objetos).

